# Hawker Centre Data Cleaning

This notebook downloads hawker centre records from `data.gov.sg` and reshapes them into a compact tabular file for feature engineering.

## Cleaning objective
- request the current hawker centre dataset
- extract names, addresses, postcodes, and coordinates
- remove duplicate records
- export the cleaned dataset to the project `data/cleaned` folder


In [1]:
# ============================================================
# 1. Imports and configuration
# ============================================================

from pathlib import Path

import pandas as pd
import requests

DATASET_ID = "d_4a086da0a5553be1d89383cd90d07ecd"
OUTPUT_FILE = Path("../../data/cleaned/hawker_centres_cleaned.csv")


/Users/celine/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Download and flatten hawker centre records

The helper function first requests a temporary download link from `data.gov.sg`, then flattens the GeoJSON response into one row per hawker centre.


In [2]:
# ============================================================
# 2. Helper function
# ============================================================

def get_hawker_centres(dataset_id: str = DATASET_ID) -> pd.DataFrame:
    """Download and flatten hawker centre records from data.gov.sg."""
    poll_url = f"https://api-open.data.gov.sg/v1/public/api/datasets/{dataset_id}/poll-download"
    poll_response = requests.get(poll_url, timeout=30)
    poll_response.raise_for_status()
    payload = poll_response.json()

    if payload.get("code", 1) != 0:
        raise RuntimeError(f"data.gov.sg error: {payload.get('errMsg')}")

    download_url = payload["data"]["url"]
    geojson_response = requests.get(download_url, timeout=30)
    geojson_response.raise_for_status()
    geojson = geojson_response.json()

    rows = []
    for feature in geojson.get("features", []):
        properties = feature.get("properties", {})
        geometry = feature.get("geometry", {})
        coordinates = geometry.get("coordinates", [None, None])

        rows.append(
            {
                "hawker_name": properties.get("NAME"),
                "address_street": properties.get("ADDRESSSTREETNAME"),
                "postal_code": properties.get("ADDRESSPOSTALCODE"),
                "lat": coordinates[1],
                "lon": coordinates[0],
            }
        )

    return pd.DataFrame(rows).drop_duplicates().reset_index(drop=True)


## Create cleaned output

This step runs the extraction, previews the cleaned dataframe, and saves the final file for downstream use.


In [3]:
# ============================================================
# 3. Build cleaned dataset
# ============================================================

hawker_df = get_hawker_centres()
print("Hawker centre records:", len(hawker_df))
hawker_df.head()


Hawker centre records: 129


,hawker_name,address_street,postal_code,lat,lon
0,Commonwealth Crescent Market,Commonwealth Crescent,149644,1.306900,103.800367
1,Tiong Bahru Market,Seng Poh Road,168898,1.284786,103.832182
2,Golden Mile Food Centre,Beach Road,199583,1.303142,103.863878
3,Yew Tee Hawker Centre,Yew Tee Close,680628,1.397185,103.745922
4,Dunman Food Centre,Onan Road,424768,1.309418,103.901825


In [4]:
# ============================================================
# 4. Save cleaned dataset
# ============================================================

OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
hawker_df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved cleaned hawker centre data to {OUTPUT_FILE.resolve()}")


Saved cleaned hawker centre data to /Users/celine/Documents/GitHub/DSA4264-Public-Policy-and-Society/data/cleaned/hawker_centres_cleaned.csv
